## FineTuning Gemma3n with Electronics repair dataset

We will finetune the latest **gemma 3n** of latest **Gemma 3** lineup of ***Google's Gemmaverse***. We will finetune both **Vision** and **Language** components. We will use `UnSloth`, `HuggingFace`, `TRL` and so on.


In [ ]:
%%capture
import os

# Check if we're in Colab and install accordingly
if "COLAB_" in "".join(os.environ.keys()):
    print(" Installing packages for Google Colab...")
    # Colab-specific installation
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth
else:
    # Local installation
    !pip install unsloth

In [ ]:
%%capture
# Install latest transformers for Gemma 3N support
!pip install --no-deps transformers==4.53.1
!pip install --no-deps --upgrade timm
!pip install wandb pillow requests


In [ ]:
# Cell 2: WandB Setup for Experiment Tracking

# Setting up Weights & Biases for tracking our fine-tuning experiments
# This helps us monitor training progress and compare different runs

import wandb

# Login to WandB
print(" Please log in to Weights & Biases for experiment tracking:")
try:
    wandb.login()
    print(" WandB login successful!")
except:
    print(" WandB login failed - training will continue without logging")
    print("You can still track progress in the notebook output")

 Please log in to Weights & Biases for experiment tracking:


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ankithreddy (ankithreddy-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


 WandB login successful!


In [ ]:
# Cell 3: HuggingFace Authentication
"""
Setting up HuggingFace Hub access for dataset loading and model uploading
"""

from huggingface_hub import login

print(" Please log in to HuggingFace Hub:")
try:
    login()
    print(" HuggingFace login successful!")
except:
    print(" HuggingFace login failed - you'll need to authenticate later for uploads")


 Please log in to HuggingFace Hub:


 HuggingFace login successful!


In [ ]:
# Cell 4: Load Base Gemma 3n Model
"""
Loading the base Gemma 3n model for demonstration
We'll use FastVisionModel for vision-language capabilities
"""

from unsloth import FastVisionModel
import torch
import gc

# Clear any existing CUDA cache
torch.cuda.empty_cache()
gc.collect()

print(" Loading Gemma 3n E2B model...")
print("This is the base model BEFORE fine-tuning on repair data")

model, processor = FastVisionModel.from_pretrained(  #  Returns model + processor
    model_name = "unsloth/gemma-3n-E2B-it",  # 2B effective parameters
    dtype = None,  # Auto-detect best dtype
    max_seq_length = 2048,  # Context length
    load_in_4bit = True,   # 4-bit quantization for memory efficiency
    use_gradient_checkpointing = "unsloth",  # Memory optimization
)

print(" Base model loaded successfully!")
print(f" Model type: {type(model)}")
print(f" Processor type: {type(processor)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
 Loading Gemma 3n E2B model...
This is the base model BEFORE fine-tuning on repair data
==((====))==  Unsloth 2025.7.8: Fast Gemma3N patching. Transformers: 4.53.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/469M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

 Base model loaded successfully!
 Model type: <class 'transformers.models.gemma3n.modeling_gemma3n.Gemma3nForConditionalGeneration'>
 Processor type: <class 'transformers.models.gemma3n.processing_gemma3n.Gemma3nProcessor'>


In [ ]:
# Cell 5: Define Inference Function
"""
Simple inference function for testing the model
"""

from transformers import TextStreamer
import gc

def run_inference(model, processor, messages, max_new_tokens=256):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        streamer=TextStreamer(processor.tokenizer, skip_prompt=True),
    )

    # Clean up memory
    del inputs, outputs
    torch.cuda.empty_cache()
    gc.collect()

print("Inference function ready")

Inference function ready


# The mutlimodel gemma 3n can see images

<img src="https://guide-images.cdn.ifixit.com/igi/YWKY5jjthyZydy2s.large" alt="Alt text" height="512">


In [ ]:
# ============================================================================
# CELL 6: Pre-Fine-tuning Demo with Embedded Instructions
# ============================================================================

# Sample Pebble Time smartwatch
pebble_image_url = "https://guide-images.cdn.ifixit.com/igi/YWKY5jjthyZydy2s.large"

print("-" * 40)

# IMPORTANT: Gemma 3N does NOT support separate system roles/prompts
# System instructions must be embedded within the user message
# This is the correct approach for Gemma 3N models
messages = [{
    "role": "user",
    "content": [
        {"type": "text", "text": "You are an expert electronics repair technician with understanding in electronics, Describe what you see in this image "},
        {"type": "image", "image": pebble_image_url},
    ]
}]

print("Question: Describe what you see in this image?")
print("\n Base model response:")
run_inference(model, processor, messages)

----------------------------------------
Question: Describe what you see in this image?

 Base model response:
## Expert Electronics Repair Technician's Assessment of the Image

This image shows a technician carefully working on the internal components of a smartwatch. Here's a detailed description of what I see, from an electronics repair perspective:

**Overall System:**

* **Smartwatch:** The device is clearly a smartwatch, likely a recent model given the components visible.
* **Disassembly:** The watch has been partially disassembled. The back cover is open, revealing the internal circuitry.
* **Repair/Troubleshooting:** The technician is using a precision screwdriver to work on a specific component, suggesting a repair or troubleshooting process.

**Visible Components and Observations:**

* **Main Circuit Board (PCB):** A small, rectangular printed circuit board is visible. It contains various electronic components.
* **Battery:** A rechargeable lithium-ion battery is present on t

In [ ]:
# ============================================================================
# CELL 7: Load Raw Dataset and Show Sample
# ============================================================================

from datasets import load_dataset

print("Loading electronics repair dataset...")
raw_dataset = load_dataset("ankithreddy/repairdataset-mini")
print(f"Loaded {len(raw_dataset['train'])} repair examples")

# Show raw sample
sample_idx = 8
raw_sample = raw_dataset['train'][sample_idx]

print("\n RAW DATASET SAMPLE:")
print("="*50)
print(f"Guide ID: {raw_sample['guide_id']}")
print(f"Device: {raw_sample['device_name']}")
print(f"Task: {raw_sample['task_title']}")
print(f"Type: {raw_sample['type']}")
print(f"Difficulty: {raw_sample['difficulty']}")
print(f"Tools: {raw_sample['tools']}")
print(f"Time: {raw_sample.get('time_required_min', 'N/A')}-{raw_sample.get('time_required_max', 'N/A')} min")
print(f"Text: {raw_sample['text'][:200]}...")
print(f"Image: {type(raw_sample['image_path'])}")


Loading electronics repair dataset...


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/103M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4649 [00:00<?, ? examples/s]

Loaded 4649 repair examples

 RAW DATASET SAMPLE:
Guide ID: 107955
Device: Polar V800
Task: Polar V800 Battery Replacement
Type: guide_overview
Difficulty: Difficult
Tools: ['T5 Torx Screwdriver', 'Spudger', 'Soldering Iron']
Time: 1800-1800 min
Text: This is a Polar V800 Battery Replacement guide. Device: Polar V800. Difficulty: Difficult. Tools needed: T5 Torx Screwdriver, Spudger, Soldering Iron. Time required: 1800-1800 minutes. Repair steps: 1...
Image: <class 'PIL.JpegImagePlugin.JpegImageFile'>


In [ ]:
# ============================================================================
# CELL 8: Filter Dataset for Step Instructions
# ============================================================================

# Filter for step instructions only
step_instructions = raw_dataset['train'].filter(lambda x: x['type'] == 'step_instruction')
guide_overviews = raw_dataset['train'].filter(lambda x: x['type'] == 'guide_overview')

print(f"Step instructions for fine-tuning: {len(step_instructions)}")
print(f"Guide overviews for RAG: {len(guide_overviews)}")

# Check what devices you have
devices = set([x['device_name'] for x in step_instructions])
print(f"Available devices: {len(devices)} unique devices")

Filter:   0%|          | 0/4649 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4649 [00:00<?, ? examples/s]

Step instructions for fine-tuning: 4158
Guide overviews for RAG: 446
Available devices: 129 unique devices


In [ ]:
# ============================================================================
# CELL 9: Create Fine-tuning Data in Chat Format
# ============================================================================

import re

def create_finetune_data(step_instructions):
    training_data = []

    instruction = "You are an expert electronics repair technician. Describe accurately what you see in this image."

    for example in step_instructions:
        # Clean the text - remove "Repair Step X:" prefixes
        cleaned_text = re.sub(r'Repair Step \d+:\s*', '', example['text'])
        cleaned_text = re.sub(r'Teardown Step \d+:\s*', '', cleaned_text)

        # Check if text starts with an -ing verb
        if re.match(r'^[a-zA-Z]+ing\s', cleaned_text.lower()):
            # Already has -ing form, use "demonstrates"
            response_text = f"This repair step for {example['task_title']} demonstrates {cleaned_text.lower()}"
        else:
            # No -ing form, use "shows how to"
            response_text = f"This repair step for {example['task_title']} shows how to {cleaned_text.lower()}"

        # Convert to chat template format
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": example['image_path']},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": response_text}]
            },
        ]

        training_example = {"messages": conversation}
        training_data.append(training_example)

    return training_data

finetune_data = create_finetune_data(step_instructions)
print(f"Created {len(finetune_data)} fine-tuning examples")

Created 4158 fine-tuning examples


In [ ]:
# ============================================================================
# CELL 10: Show Fine-tuning Data Samples
# ============================================================================

for i in range(3):
   example = finetune_data[i]
   print(f"EXAMPLE {i+1}:")
   print(f"USER: {example['messages'][0]['content'][0]['text']}")
   print(f"ASSISTANT: {example['messages'][1]['content'][0]['text'][:150]}...")
   print(f"IMAGE: {type(example['messages'][0]['content'][1]['image'])}")
   print("-" * 50)

print(f"Total examples: {len(finetune_data)}")

EXAMPLE 1:
USER: You are an expert electronics repair technician. Describe accurately what you see in this image.
ASSISTANT: This repair step for Microsoft Band Wrist Clasp Replacement demonstrates using the t3 torx screwdriver, remove the two 2.00mm long screws from the ins...
IMAGE: <class 'PIL.JpegImagePlugin.JpegImageFile'>
--------------------------------------------------
EXAMPLE 2:
USER: You are an expert electronics repair technician. Describe accurately what you see in this image.
ASSISTANT: This repair step for Microsoft Band Wrist Clasp Replacement shows how to remove the metal plate with the serial and model number on it by pulling it s...
IMAGE: <class 'PIL.JpegImagePlugin.JpegImageFile'>
--------------------------------------------------
EXAMPLE 3:
USER: You are an expert electronics repair technician. Describe accurately what you see in this image.
ASSISTANT: This repair step for Microsoft Band Wrist Clasp Replacement shows how to pull the two securing pins out of the ru

In [ ]:
# Cell 9: Setup LoRA for Fine-tuning
"""
Preparing model for parameter-efficient fine-tuning with LoRA
"""

print("Setting up LoRA...")

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    target_modules="all-linear",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

print(" LoRA setup complete")

Setting up LoRA...


RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [ ]:
# Cell 10: Configure Trainer
"""
Setup trainer for fine-tuning
"""

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

print("Configuring trainer...")

trainer = SFTTrainer(
    model=model,
    train_dataset=finetune_data,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor, resize=512),
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_grad_norm=0.3,
        warmup_steps=15,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="steps",
        save_steps=50,
        optim="adamw_torch_fused",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="repair_assistant_checkpoints",
        run_name="lerobot-repairbot-assistant-v1",
        report_to="wandb",

        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)

print(" Trainer configured")

Configuring trainer...
 Trainer configured


In [ ]:
# ============================================================================
# CELL 12: Display Memory Stats Before Training
# ============================================================================

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Memory reserved before training: {start_gpu_memory} GB")

GPU: Tesla T4
Max memory: 14.741 GB
Memory reserved before training: 10.988 GB


In [ ]:
# Cell 13: Start Training
"""
Begin fine-tuning the model
"""

print(" Starting training...")

trainer_stats = trainer.train()

print(" Training complete!")

# Display final training stats
final_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
memory_used = round(final_gpu_memory - start_gpu_memory, 3)

print(f"\n TRAINING SUMMARY:")
print(f"   • Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"   • Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"   • Peak memory usage: {final_gpu_memory} GB")
print(f"   • Memory used for training: {memory_used} GB")

 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,158 | Num Epochs = 1 | Total steps = 1,040
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,135,360 of 5,460,573,632 (0.39% trained)


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,10.053100
10,9.882200
15,11.049200
20,5.192700
25,2.580100
30,2.496200
35,2.001800
40,1.847800
45,2.027200
50,1.940200


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


 Training complete!

 TRAINING SUMMARY:
   • Training time: 112.98 minutes
   • Final loss: 1.3572852964584643
   • Peak memory usage: 10.988 GB
   • Memory used for training: 0.0 GB


<img src="https://guide-images.cdn.ifixit.com/igi/T5IFtANCaFRiUVYI.large" alt="Alt text" height="512">

In [ ]:
# ============================================================================
# CELL 14: Post-Training Inference Test
# ============================================================================

print("\n TESTING FINE-TUNED MODEL:")
print("="*50)

instruction = "You are an expert electronics repair technician. Describe accurately what you see in this image."
image = "https://guide-images.cdn.ifixit.com/igi/T5IFtANCaFRiUVYI.large"
# Set model to inference mode
FastVisionModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": [{"type": image}, {"type": "text", "text": instruction}],
    }
]

print("\n FINE-TUNED Model response:")
run_inference(model, processor, messages, max_new_tokens=300)



 TESTING FINE-TUNED MODEL:

 FINE-TUNED Model response:
## Expert Electronics Repair Technician Analysis of the Image

This image shows a close-up view of a **Samsung Galaxy Watch4 Classic Battery Replacement** performed using a **heat gun** to loosen the adhesive holding the battery in place. the watch is being held by a **tweezers** while the heat gun is being used to loosen the adhesive. the watch is being carefully lifted out of the watch frame. the battery is being carefully held by the tweezers. the watch is being carefully turned over to expose the battery. the battery is being carefully removed from the watch frame. the battery is being carefully placed back into the watch frame. the watch is being carefully closed. the watch is being carefully turned back over. the watch is being carefully lifted out of the watch frame. the watch is being carefully closed. the watch is being carefully turned back over. the watch is being carefully lifted out of the watch frame. the watch is b

Saving , loading fintuned models